<a href="https://colab.research.google.com/github/Davidson-Open-University/phy107-labs/blob/main/notebooks/PHY107-Lab4-Oscillations-and-Resonance.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PHY 107 Lab 4: Oscillations and Resonance

**Unit 4: Electrical and mechanical resonant systems** · General Practical Physics I · Davidson Open University

**In this lab you will:**
- measure a period by timing many oscillations;
- use a pendulum and a mass–spring system to find g and k;
- calculate the resonant frequency of an LC circuit;
- find the resonance peak in frequency-response data.

**Time:** about 60 minutes in four parts:
- **Part A – Explore** (about 10 minutes): run the online lab and the examples, and read the explanations.
- **Part B – Practise** (about 20 minutes): fill in the blanks. A self-check tells you if you are right.
- **Part C – Build a tool** (about 20 minutes): complete a short analysis function and test it on data.
- **Part D – Your own data and reflection** (about 10 minutes): analyse readings you took in the online lab (or at home), and write short answers.

**How to use this notebook**
1. Run each code cell in order: click it and press **Shift + Enter** (or the ▶ button).
2. Where you see `...`, replace it with your own code, run the cell, then run the **check** cell under it.
   - ✅ means correct. ❌ gives you a hint: try again. ⏳ means you have not filled in the blank yet.
   - The checks never show the answer, so you learn by working it out. You can try as many times as you like.
3. Stuck? Read the hint, look back at the study session, or ask in the discussion forum. Do not skip ahead: each part builds on the last.
4. Save your work: in Colab choose **File → Save a copy in GitHub** and save it to your own `phy107-labs` repository. Offline in Jupyter, press **Ctrl + S** and later upload the file with **Add file → Upload files** on GitHub.

**Honest data:** keep your own readings separate from the example data, and label every simulated or supplied reading as such. Never invent readings.

*This lab is practice to help you learn and prepare for the unit assignment (lab report). It is not graded unless your tutor tells you otherwise.*

In [ ]:
#@title ▶ Run this cell first: it sets up the self-checks (you do not need to read it)
# You do not need to read or change this code.
import hashlib as _h
class _Blank(Exception): pass
def _norm(x, unordered=False, dp=6):
    try:
        import numpy as _np
        if isinstance(x, _np.ndarray): x = x.tolist()
        elif isinstance(x, _np.bool_): x = bool(x)
        elif isinstance(x, _np.integer): x = int(x)
        elif isinstance(x, _np.floating): x = float(x)
    except ImportError:
        pass
    if x is Ellipsis or x is None: raise _Blank()
    if isinstance(x, bool): return 'b:' + str(x)
    if isinstance(x, (int, float)): return 'n:' + repr(round(float(x), dp) + 0.0)
    if isinstance(x, str): return 't:' + ' '.join(x.lower().split())
    if isinstance(x, (set, frozenset)): return 's{' + ','.join(sorted(_norm(i, False, dp) for i in x)) + '}'
    if isinstance(x, (list, tuple)):
        items = [_norm(i, False, dp) for i in x]
        if unordered: items.sort()
        return 'l[' + ','.join(items) + ']'
    if isinstance(x, dict): return 'd{' + ','.join(sorted(_norm(k, False, dp) + '=' + _norm(v, False, dp) for k, v in x.items())) + '}'
    try:
        import sympy as _sp
        if isinstance(x, _sp.Basic):
            x = _sp.sympify(x)
            if x.is_number: return 'n:' + repr(round(float(x), dp) + 0.0)
            syms = sorted(x.free_symbols, key=str)
            vals = []
            for t in (0.3, 0.7, 1.3):
                v = complex(x.subs({s: t + 0.11 * i for i, s in enumerate(syms)}).evalf())
                vals.append(round(v.real, 6) + 0.0)
            return 'e:' + repr(vals)
    except ImportError:
        pass
    return 'r:' + repr(x)
def _hash(q, s): return _h.sha256(('phy107|' + q + '|' + s).encode()).hexdigest()[:16]
def check(q, value):
    """Check your answer to question q. Prints ✅, ❌ with a hint, or ⏳ if the blank is not filled yet."""
    spec = _CHECKS[q]
    try:
        if callable(value) and 'args' in spec:
            outs = []
            for a in spec['args']:
                try:
                    outs.append(value(*a))
                except _Blank:
                    raise
                except Exception as e:
                    print(f'❌ Question {q}: your function stopped with an error: {type(e).__name__}: {e}')
                    return
            if all(o is None or o is Ellipsis for o in outs): raise _Blank()
            got = _norm(outs, False, spec.get('dp', 6))
        elif spec.get('logic'):
            if value is None or value is Ellipsis: raise _Blank()
            import sympy as _sp
            from itertools import product as _p
            syms = sorted(_sp.sympify(value).free_symbols, key=str)
            got = 'L' + ''.join('1' if bool(_sp.sympify(value).subs(dict(zip(syms, v)))) else '0' for v in _p([True, False], repeat=len(syms))) + ':' + ','.join(map(str, syms))
        else:
            got = _norm(value, spec.get('unordered', False), spec.get('dp', 6))
    except _Blank:
        print(f'⏳ Question {q}: fill in the blank (replace ...) and run the cell above, then run this cell again.')
        return
    if _RECORD is not None:
        _RECORD[q] = _hash(q, got); print(f'✅ Question {q}: recorded.'); return
    if _hash(q, got) == spec['h']:
        print(f'✅ Question {q}: correct! ' + spec.get('ok', 'Well done.'))
    else:
        print(f'❌ Question {q}: not yet. Hint: ' + spec['hint'])
_RECORD = None

_CHECKS = {
    '4.1': {'h': '69fdf30e54d8d496', 'hint': 'Period = total time / number of oscillations.', 'ok': 'Timing many oscillations divides the reaction-time error by 20.', 'dp': 2},
    '4.2': {'h': '552985d0c8331729', 'hint': '4 * np.pi**2 * L / T**2', 'dp': 2},
    '4.3': {'h': '5508419de89b41ac', 'hint': 'T = 2 * np.pi * np.sqrt(m / k)', 'dp': 3},
    '4.4': {'h': '146b4c6db0cd36dc', 'hint': '1 / (2 * np.pi * np.sqrt(L * C)), with L and C in henries and farads.', 'ok': 'About 1.6 kHz: this is how a radio tuner selects a station.', 'dp': 0},
    '4.5': {'h': '80dddfb31c597b68', 'hint': 'np.argmax(amp) gives the position of the largest amplitude; use it to index freq.', 'ok': 'More damping would make this peak lower and wider.', 'dp': 2},
    '4.6': {'h': 'f651a854df0db158', 'hint': 'Convert to arrays, gradient = np.polyfit(L, T**2, 1)[0], return round(4 * np.pi**2 / gradient, 2).', 'ok': 'The graph method uses every reading.', 'dp': 2, 'args': [([0.2, 0.4, 0.6, 0.8, 1.0], [0.899, 1.268, 1.556, 1.795, 2.007]), ([0.5, 1.0, 1.5], [1.42, 2.01, 2.46])]},
}
print('Self-checks ready. Now carry on with Part A.')

## Part A – Explore (about 10 minutes)
### 🔬 Online lab for this unit (free open educational resources)
- **[PhET Pendulum Lab](https://phet.colorado.edu/en/simulations/pendulum-lab)** – change length and mass; time oscillations with the stopwatch and period timer.
- **[PhET Masses and Springs](https://phet.colorado.edu/en/simulations/masses-and-springs)** – hang masses on springs and watch the oscillation and damping.
- **[PhET Circuit Construction Kit: AC](https://phet.colorado.edu/en/simulations/circuit-construction-kit-ac)** – build an inductor–capacitor circuit and explore how current oscillates.

Open the online lab in a new tab, try it for a few minutes, then come back here. **Label every reading you take there as "simulated".**

**Key results.** Pendulum: T = 2π√(L/g). Mass on a spring: T = 2π√(m/k). Inductor–capacitor circuit: resonant frequency f₀ = 1 / (2π√(LC)). **Resonance** happens when a system is driven at its natural frequency, giving the biggest response.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
for L in [0.25, 0.5, 1.0]:
    print(f'L = {L} m  ->  T = {2 * np.pi * np.sqrt(L / 9.81):.3f} s')

## Part B – Practise (about 20 minutes)
**B1.** 20 oscillations take 28.4 s. What is the period T in seconds (2 decimal places)?

In [ ]:
answer_4_1 = ...

In [ ]:
check('4.1', answer_4_1)

In [ ]:
answer_4_2 = ...   # g from L = 0.50 m and T = 1.42 s, using g = 4π²L/T², 2 decimal places

In [ ]:
check('4.2', answer_4_2)

In [ ]:
answer_4_3 = ...   # T for m = 0.20 kg on a spring with k = 20 N/m, 3 decimal places

In [ ]:
check('4.3', answer_4_3)

**B4. LC resonance.** Find f₀ in hertz for L = 10 mH (0.010 H) and C = 1.0 µF (1.0e-6 F). Round to the nearest whole number.

In [ ]:
answer_4_4 = ...

In [ ]:
check('4.4', answer_4_4)

**B5. Finding the peak.** The illustrative data below give the amplitude of a driven oscillator at different driving frequencies. Find the driving frequency (Hz) with the **largest** amplitude. *Hint:* `freq[np.argmax(amp)]`.

In [ ]:
freq = np.array([0.5, 0.8, 1.0, 1.1, 1.2, 1.3, 1.4, 1.6, 2.0])     # Hz  (illustrative)
amp = np.array([1.1, 1.6, 2.6, 4.0, 6.2, 4.4, 2.9, 1.7, 0.9])      # cm
plt.plot(freq, amp, 'o-'); plt.xlabel('driving frequency (Hz)'); plt.ylabel('amplitude (cm)'); plt.show()

In [ ]:
answer_4_5 = ...

In [ ]:
check('4.5', answer_4_5)

## Part C – Build a tool (about 20 minutes)
Complete `g_from_pendulum(lengths, periods)`. It should fit **T² against L** and return g = 4π² / gradient, rounded to 2 decimal places.

In [ ]:
def g_from_pendulum(lengths, periods):
    ...      # YOUR CODE

In [ ]:
check('4.6', g_from_pendulum)

## Part D – Your own data and reflection (about 10 minutes)
**D1. Prepares you for the Unit 4 assignment.** In PhET Pendulum Lab, time 10 oscillations for at least five lengths, then use your tool to find g. Or make a safe string pendulum at home (a key on a 1 m string, small swings, nothing fragile nearby) and time it with a phone stopwatch. Label simulated and home data clearly.

In [ ]:
my_lengths = [...]
my_periods = [...]
# print(g_from_pendulum(my_lengths, my_periods))

**D2. Reflect.**
1. Why must pendulum swings be small for T = 2π√(L/g) to hold?
2. What does damping do to a resonance peak?
3. Name one useful and one harmful example of resonance.

*Type your answer here (double-click this cell to edit it).*

## Finished? Save your work
In Colab: **File → Save a copy in GitHub** → your `phy107-labs` repository → commit message *"Lab 4 completed"*.
Offline: save with **Ctrl + S**, then upload the file to your repository with **Add file → Upload files** when you are online.

Then go back to Moodle and mark the lab as done.